# L28 Lab: FMCW Range, notebook

This notebook runs the course's radar chain, `radar_lab.py`, on your saved captures. You do not write the chain; you run it, change what each task names, and record what you see in Table E of the L28 Lab packet. There are three tasks, (a) to (c), about 30 to 45 minutes in all, and each one changes one line.

**To run it:** put this notebook, `radar_lab.py`, and your captures in one folder, or keep the course's layout (`notebooks/` beside `radar-data/`). It needs Python 3 with numpy, scipy, and matplotlib, and any Jupyter (JupyterLab, VS Code, or the one the course provides). Run the cells in order from the top.

If your own captures failed, the course's recorded data (`L28_*.npz`) works in every cell.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import radar_lab as rl

# Where the captures are: the course's recorded data if this folder sits beside
# radar-data/, otherwise this folder.
DATA = Path("../radar-data") if Path("../radar-data").exists() else Path(".")

# Your 2 m capture (Procedure B). Change the name to your own file to use your data.
CAPTURE = DATA / "L28_corner_2m.npz"
cap = rl.load_capture(CAPTURE)
print(f"{CAPTURE.name}: {cap['iq'].shape[0]} chirps of {cap['iq'].shape[1]} samples, "
      f"B = {cap['B']/1e6:.0f} MHz, T_ramp = {cap['T_ramp']*1e3:.2f} ms, f0 = {cap['f0']/1e3:.1f} kHz")

In [ ]:
# Readouts used by the tasks. Run this cell; you do not need to change it.
def peak(r, p_db, lo=0.5, hi=8.0):
    """Range and level of the strongest point between lo and hi meters."""
    m = (r >= lo) & (r <= hi)
    i = np.flatnonzero(m)[np.argmax(p_db[m])]
    return i, r[i], p_db[i]

def width_3db(r, p_db, i):
    """Width in meters of the peak at index i, 3 dB down."""
    lo = hi = i
    while lo > 0 and p_db[lo - 1] > p_db[i] - 3: lo -= 1
    while hi < len(p_db) - 1 and p_db[hi + 1] > p_db[i] - 3: hi += 1
    return r[hi] - r[lo] + (r[1] - r[0])

def sidelobe(r, p_db, i, span=1.2):
    """Strongest sidelobe within span m of the peak at index i, in dB below it."""
    a = b = i
    while a + 1 < len(p_db) and p_db[a + 1] < p_db[a]: a += 1     # walk down to each first null
    while b > 0 and p_db[b - 1] < p_db[b]: b -= 1
    near = (np.abs(r - r[i]) < span)
    out = near.copy(); out[b:a + 1] = False
    return p_db[out].max() - p_db[i]

## Task (a): Hann against rectangular

The cell below draws one chirp's range profile and reads three numbers: the plate's peak, its strongest sidelobe, and the floor near 10 m.

1. Run it as it stands (`WINDOW = "hann"`) and record the three numbers in the Hann column of Table E.
2. **Change one line:** set `WINDOW = None`, the rectangular window, and run it again. Record the rectangular column.

**Check:** with Hann the strongest sidelobe is about 31 dB below the peak; with the rectangular window it is about 13 dB below. The peak level hardly changes, because `radar_lab` divides every FFT by the sum of its window. The floor near 10 m rises by about 20 dB with the rectangular window. Look at the plot and decide whether that rise is noise or something else; Part 3 of the packet asks you.

In [ ]:
WINDOW = "hann"          # TASK (a): change to None for the rectangular window
N_FFT = 4096             # zero-padded, so the peak is easy to read

r, p_db = rl.range_profile(cap, chirp=0, window=WINDOW, n_fft=N_FFT)
i, r_pk, p_pk = peak(r, p_db)
floor = np.median(p_db[(r > 8) & (r < 12)])
print(f"window {WINDOW}: peak {r_pk:.2f} m at {p_pk:.1f} dBFS, "
      f"strongest sidelobe {sidelobe(r, p_db, i):.1f} dBc, floor near 10 m {floor:.1f} dBFS")

plt.figure(figsize=(8, 3))
plt.plot(r, p_db)
plt.xlim(0, 12); plt.ylim(-100, 0)
plt.xlabel("range (m)"); plt.ylabel("power (dBFS)"); plt.title(f"one chirp, window {WINDOW}")
plt.grid(alpha=0.3); plt.show()

## Task (b): does zero-padding resolve two targets?

`L28_two_corners.npz` holds two corners 0.20 m apart, at 2.0 and 2.2 m: closer than the 0.30 m resolution cell. The cell draws them with no padding, 300 points across 0 to 30 m, one per 0.30 m.

1. Run it as it stands (`N_FFT = None`) and count the peaks between 1.5 and 2.7 m.
2. **Change one line:** set `N_FFT = 4096` and run it again. Count the peaks and record the width 3 dB down.

**Check:** one peak both times, about 0.5 m wide with Hann, a little wider than one corner alone (0.45 m). Zero-padding draws the same spectrum on a grid 14 times finer; it cannot split two beats that differ by less than one cycle a chirp. Record the width in the last row of Table E.

In [ ]:
N_FFT = None             # TASK (b): change to 4096

two = rl.load_capture(DATA / "L28_two_corners.npz")
r2, p2 = rl.range_profile(two, window="hann", n_fft=N_FFT)
m = (r2 > 1.5) & (r2 < 2.7)
peaks = np.flatnonzero(m[1:-1] & (p2[1:-1] > p2[:-2]) & (p2[1:-1] > p2[2:]) & (p2[1:-1] > p2[m].max() - 6)) + 1
i2, r_pk2, _ = peak(r2, p2, 1.5, 2.7)
print(f"n_fft {N_FFT}: {len(peaks)} peak(s) between 1.5 and 2.7 m, at {np.round(r2[peaks], 2)} m; "
      f"width 3 dB down {width_3db(r2, p2, i2):.2f} m")

plt.figure(figsize=(8, 3))
plt.plot(r2, p2, ".-" if N_FFT is None else "-")
plt.axvline(2.0, color="gray", ls=":"); plt.axvline(2.2, color="gray", ls=":")
plt.xlim(0.5, 4); plt.ylim(-80, 0)
plt.xlabel("range (m)"); plt.ylabel("power (dBFS)"); plt.title(f"two corners 0.20 m apart, n_fft {N_FFT}")
plt.grid(alpha=0.3); plt.show()

## Task (c): range from your own 0 m frequency

`range_profile` converts frequency to range for you, using the capture's `f0`. This task does the conversion by hand, from the frequency of the plate's peak, with **your** 0 m frequency from Table A.

1. Set `F0` to your Table A value, in hertz.
2. **Change one line:** replace `None` in `R = None` with the range of the peak, from `f_peak`, `F0`, and the capture's `B` and `T_ramp` (the range equation of the lesson's range FFT frame). Use `rl.C` for the speed of light.

**Check:** `R` agrees with the tape-measure range of the capture to within about 0.15 m, and with the peak the profile in task (a) found. Record `R` in Table E. If it is off by a fixed amount at every range, look at `F0` first: 1 kHz of error in $f_0$ moves every range by 0.15 m.

In [ ]:
F0 = cap["f0"]           # TASK (c), step 1: your Table A 0 m frequency, in Hz

X = rl.range_fft(cap, window="hann", n_fft=4096)[0]
f = np.fft.fftfreq(4096, 1 / cap["fs"])
search = (f > F0 + 3e3) & (f < cap["fs"] / 2)        # beats beyond about 0.5 m
f_peak = f[search][np.argmax(np.abs(X[search]))]

R = None                 # TASK (c), step 2: the range of the peak, in m

print(f"peak at {f_peak/1e3:.2f} kHz, {(f_peak - F0)/1e3:.2f} kHz above F0; R = {R}")

## Your waterfall

No task here: run it on your walk capture (Procedure D) to make the waterfall plot the packet asks for. Change `WALK` to your own file.

In [ ]:
WALK = DATA / "L28_walker.npz"
walk = rl.load_capture(WALK)
rw, tw, w_db = rl.waterfall(walk)
keep = rw <= 6
plt.figure(figsize=(8, 4))
plt.pcolormesh(rw[keep], tw, w_db[:, keep], vmin=-65, vmax=-25, cmap="Blues", shading="auto")
plt.xlabel("range (m)"); plt.ylabel("time (s)"); plt.title(WALK.name)
plt.colorbar(label="dBFS"); plt.show()

## Before you close

Save the notebook with your changes and your plots, and turn it in with the packet. Keep it, `radar_lab.py`, and your captures together: the L29 Lab's notebook starts from them. Anything you do beyond these three tasks, such as your own plot, another window, or another capture, belongs in your report's Extra Effort.